# 系统风险看板-模块二 波动率与尾部风险

In [ ]:
# Cell1：导入包
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

In [ ]:
# Cell2：获取指数行情
index_codes = ["000300.XSHG","000905.XSHG"]
# 5年日线数据，count=5*220，panel=False 返回长DataFrame
df_price = get_price(
    index_codes, 
    end_date=datetime.now(),
    count=5*252+60,
    frequency="daily",
    fields=["close"],
    skip_paused=False,
    fq='pre',
    panel=False,
    fill_paused=True
)

# 透视：行=日期，列=指数代码，值=收盘价（panel=False长表转宽表，方便计算）
df_close = df_price.pivot(index="time", columns="code", values="close")
df_close.index = pd.to_datetime(df_close.index)
print("收盘价表形状：", df_close.shape)
df_close.head()

In [ ]:
# Cell3：计算对数收益率
# 对数收益率 r = ln(Pt / Pt-1)
ret = np.log(df_close / df_close.shift(1))
ret = ret.dropna()
print("收益率表：")
ret.head()

In [ ]:
# Cell4：核心指标计算：20d/60d 波动率、波动率斜率、下行半方差
# 年化系数 252交易日
annual = np.sqrt(252)
win20 = 20
win60 = 60

# 已实现波动率（年化）
vol20 = ret.rolling(window=win20).std() * annual
vol60 = ret.rolling(window=win60).std() * annual

# 波动率斜率 = 20日波动率 / 60日波动率
vol_slope = vol20 / vol60

# ========= 下行半方差：仅r<0的收益率计算方差 =========
def rolling_downside_var(series, window):
    """
    下行半方差：只取负收益样本
    series: 单指数收益率Series
    """
    def downside_var_window(arr):
        arr = arr[~np.isnan(arr)]
        neg_ret = arr[arr < 0]
        if len(neg_ret) < 3:
            return np.nan
        return np.var(neg_ret, ddof=1)
    return series.rolling(window=window).apply(downside_var_window, raw=True)

# 逐列计算下行半方差（窗口60日）
downside_var = pd.DataFrame(index=ret.index, columns=ret.columns)
for col in ret.columns:
    downside_var[col] = rolling_downside_var(ret[col], window=60)

# 下行波动率 = sqrt(下行半方差) 年化
downside_vol = np.sqrt(downside_var) * annual

# 合并基础指标面板
df_indicators = pd.concat({
    "vol20": vol20,
    "vol60": vol60,
    "vol_slope": vol_slope,
    "downside_var": downside_var,
    "downside_vol": downside_vol
}, axis=1)

df_indicators.head()


In [ ]:
# Cell5：滚动历史模拟 VaR & CVaR（95% 置信度，60 日滚动窗口）
# 历史模拟法 VaR & CVaR，置信水平95%，窗口60
conf_level = 0.05
win_var = 60

def rolling_var(series, window, alpha=0.05):
    """单独计算VaR，返回标量"""
    def calc_var(arr):
        arr = arr[~np.isnan(arr)]
        if len(arr) < window * 0.8:
            return np.nan
        var = np.percentile(arr, alpha*100)
        return var
    return series.rolling(window=window).apply(calc_var, raw=True)


def rolling_cvar(series, window, alpha=0.05):
    """单独计算CVaR，返回标量"""
    def calc_cvar(arr):
        arr = arr[~np.isnan(arr)]
        if len(arr) < window * 0.8:
            return np.nan
        var = np.percentile(arr, alpha*100)
        tail = arr[arr <= var]
        cvar = np.mean(tail)
        return cvar
    return series.rolling(window=window).apply(calc_cvar, raw=True)

# 存放结果
var_df = pd.DataFrame(index=ret.index, columns=ret.columns)
cvar_df = pd.DataFrame(index=ret.index, columns=ret.columns)

for col in ret.columns:
    var_df[col] = rolling_var(ret[col], window=win_var, alpha=conf_level)
    cvar_df[col] = rolling_cvar(ret[col], window=win_var, alpha=conf_level)

# 合并VaR/CVaR到指标总表
df_indicators[("VaR_95", "000300.XSHG")] = var_df["000300.XSHG"]
df_indicators[("VaR_95", "000905.XSHG")] = var_df["000905.XSHG"]

df_indicators[("CVaR_95", "000300.XSHG")] = cvar_df["000300.XSHG"]
df_indicators[("CVaR_95", "000905.XSHG")] = cvar_df["000905.XSHG"]

df_indicators.head()


In [ ]:
# Cell6：计算 5 年滚动历史分位数（匹配风控看板分位体系）
def rolling_percentile_rank(series, window):
    """
    滚动分位：当前值在滚动窗口内的百分位排名，0~1
    兼容老pandas
    """
    def rank_func(arr):
        arr = arr[~np.isnan(arr)]
        val = arr[-1]
        if np.isnan(val):
            return np.nan
        pct = np.sum(arr <= val) / len(arr)
        return pct
    return series.rolling(window=window).apply(rank_func, raw=True)

# 分位窗口：5年交易日 1260
win_pctl = 1260

# 更新指标列表，匹配新的多级列名
target_indicators = [
    ("vol20", "000300.XSHG"),
    ("vol20", "000905.XSHG"),
    ("vol_slope", "000300.XSHG"),
    ("vol_slope", "000905.XSHG"),
    ("downside_vol", "000300.XSHG"),
    ("downside_vol", "000905.XSHG"),
    ("VaR_95", "000300.XSHG"),
    ("VaR_95", "000905.XSHG"),
    ("CVaR_95", "000300.XSHG"),
    ("CVaR_95", "000905.XSHG"),
]

# 新建分位DataFrame
df_pctl = pd.DataFrame(index=df_indicators.index)
for col_tuple in target_indicators:
    s = df_indicators[col_tuple]
    name = "_".join(col_tuple)
    df_pctl[name] = rolling_percentile_rank(s, win_pctl)

print("分位结果预览")
df_pctl.dropna().head()

In [ ]:
# 检查cell4生成指标的非空数量
print("vol20非空数量：", df_indicators['vol20']['000300.XSHG'].notna().sum())
print("downside_vol非空数量：", df_indicators['downside_vol']['000300.XSHG'].notna().sum())
print("VaR非空数量：", var_df['000300.XSHG'].notna().sum())
print("CVaR非空数量：", cvar_df['000300.XSHG'].notna().sum())
print("df_pctl非空行数：", df_pctl.dropna().shape[0])


In [ ]:
# ========= 提取最新一日数据 =========
# 取最新一行指标
latest_indicators = df_indicators.iloc[-1]
# 取最新一行分位
latest_pctl = df_pctl.iloc[-1]

# 展平指标
res_list = []
for idx in latest_indicators.index:
    cat, code = idx
    val = latest_indicators[idx]
    # 匹配对应的分位列名
    pctl_name = f"{cat}_{code}"
    if pctl_name in latest_pctl.index:
        pctl_val = latest_pctl[pctl_name]
    else:
        pctl_val = np.nan
    res_list.append({
        "指数": code,
        "指标名称": cat,
        "指标原值": val,
        "5年历史分位(0~1)": pctl_val
    })

df_module2_summary = pd.DataFrame(res_list)

# 打印汇总表
print("===== 模块二【波动率与尾部风险】最新交易日指标汇总 =====")
display(df_module2_summary)

# 提取用于风险加权打分的【沪深300】各指标分位（你看板主基准）
# 你原始看板模块二权重是25%，后续综合总分直接取用这几个分位
hs300_pctl = df_module2_summary[df_module2_summary["指数"]=="000300.XSHG"].set_index("指标名称")["5年历史分位(0~1)"]
print("\n===== 沪深300 模块二分位（供综合风险总分计算） =====")
display(hs300_pctl)


 逐项拆解（沪深 300）

1. **vol20（20 日已实现年化波动率）：0.53**
当前 20 日波动率处在 5 年历史的**53% 分位**。中等波动水平，不算高风险。
2. **vol60：NaN**
我前面写的权重字典里**没有纳入 vol60**，所以它没有参与打分，属于正常。vol60 只是用来计算波动率斜率的中间指标，不作为独立风险因子。
3. **vol_slope（20d/60d 波动率斜率）：0.094**
很低的分位。代表短期波动率相比长期波动率**没有抬升**，不存在风险加速爆发的特征。
    - 波动率斜率冲高（高分位）一般是恐慌快速升温的信号，当前这个指标很安全。
4. **downside_var：NaN**
下行方差是中间计算变量；我们用的是`downside_vol`下行波动率，所以`downside_var`不参与打分，正常。
5. **downside_vol（下行年化波动率）：0.866** ✅重点
**很高的分位！**
下行波动率，只统计下跌阶段波动。说明：**市场向下的波动，处于 5 年里偏高区间**。
    - 这里是当前模块二里最主要的风险贡献项。虽然整体波动率中等，但下跌的时候波动偏大。
6. **VaR_95：0.047**
95% 置信度收益率 VaR（单日尾部损失）处于很低历史分位，极端单日大跌的概率在历史上偏低。
7. **CVaR_95：0.222**
条件风险价值，代表一旦突破 VaR 之后，尾部平均损失。当前分位不高，极端崩盘尾部损失预期不大。

In [ ]:
# 模块二内部加权：你可以按需调整权重
weight_map = {
    "vol20":0.25,
    "vol_slope":0.15,
    "downside_vol":0.30,
    "VaR_95":0.15,
    "CVaR_95":0.15
}

score = 0.0
for name,w in weight_map.items():
    score += hs300_pctl[name] * w

module2_total_score = score * 100
print(f"\n模块二【波动率尾部风险】综合得分(0~100)：{module2_total_score:.2f}")


含义对照你四层风险档位

- 0~40：绿色舒适区
- 40~65：黄色审慎区
- 65~85：橙色防御区
- 85~100：红色危机区

👉 **模块二当前落在【审慎区（黄色）】**

> 
> 解读总结：
> 整体尾部风险中等。**亮点：波动率斜率、VaR、CVaR 都很低，没有爆发式恐慌；但短板是下行波动率偏高，下跌阶段的波动压力不小。**
> 这个 44.73，后续乘以全局权重 25%，贡献到总的综合风险分数。

潜在优化小建议

`downside_vol`分位很高但 vol20 一般，这说明**涨跌波动不对称，跌的时候波动大，上涨波动不大**。

如果你想，可以考虑：

- 校验下行半方差计算逻辑，确认是否是近期小幅持续下跌带来这个高下行波动率；
- 或者给 downside_vol 设置一个封顶，防止它单独把模块分数拉高。

In [ ]:
# Cell7：绘图模块
# 绘图示例：画沪深300 20d波动率和波动率斜率
fig, axes = plt.subplots(2,1,figsize=(14,8))

# 子图1：20日年化波动率
axes[0].plot(df_indicators["vol20"]["000300.XSHG"], label="沪深300 vol20")
axes[0].plot(df_indicators["vol20"]["000905.XSHG"], label="中证500 vol20")
axes[0].set_title("20日已实现年化波动率")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 子图2：波动率斜率 vol20/vol60
axes[1].plot(df_indicators["vol_slope"]["000300.XSHG"], label="沪深300 vol_slope")
axes[1].plot(df_indicators["vol_slope"]["000905.XSHG"], label="中证500 vol_slope")
axes[1].set_title("波动率斜率 vol20/vol60")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


上图：20 日已实现年化波动率 vol20

这个指标代表**过去 20 个交易日价格波动的年化水平**，数值越高，短期震荡越大。

1. **整体特征**：中证 500 波动率几乎全程高于沪深 300。小盘股天然波动更大，完全符合 A 股特性。
2. **历史峰值**
   - 2022、2024、2025 出现多次尖峰，是市场快速急跌 / 剧烈震荡阶段，波动率脉冲式飙升。
   - 尖峰特点：**快速冲高、快速回落**，恐慌波动大多是短期脉冲。
3. **当前末端（2026 右侧）**
   - 沪深 300 vol20 回落；中证 500 还处在一波上行后的回落阶段。
   - 对应刚才计算的分位：沪深 300 vol20 在 53% 分位，处于历史中等波动区间，**没有极端恐慌，但也不是低波动舒适区**。

下图：波动率斜率 vol_slope = vol20 /vol60

> 
> 含义：短期波动率 ÷ 长期波动率

- **斜率 >1**：短期波动率 > 长期波动率 → 波动正在**加速抬升**，市场恐慌在快速升温，是风险预警信号；
- **斜率 <1**：短期波动率 < 长期波动率 → 短期波动相比长期基准在收敛，没有爆发式恐慌。

1. **历史形态**
每次市场危机 / 大跌，都会看到斜率瞬间冲到 1.4~1.6 的高点（尖峰），代表短期波动急剧放大；
底部位置可以低到 0.4~0.6，属于波动快速消退后的阶段。
2. **当前末端（2026 最右侧）**
沪深 300、中证 500 斜率**明显向下，跌到 1 以下**。
对应刚才算出的沪深 300 vol_slope 分位仅 0.09：**在 5 年历史里处于很低区间**。
✅ 解读：**没有短期波动加速的迹象，不存在恐慌快速发酵的特征**。
哪怕前面波动率有一波抬升，现在短期波动已经弱于长期波动，风险脉冲正在消退。

👉 组合结论：
**最近一段时间，上涨的时候波动不大，但一旦下跌，回撤的震荡幅度偏高（下跌不对称风险）；但是没有出现短期恐慌加速爆发（波动率斜率很低）。**
也就是：不是那种一次性崩盘式的急跌危机，而是 “阴跌、下跌过程震荡大” 的环境。

风控层面的实战含义

1. 波动率斜率是**前置预警指标**，危机来临前往往率先向上突破 > 1。现在斜率回落，**没有马上爆发剧烈股灾的信号**；
2. 但下行波动率偏高，意味着**下行端的脆弱性存在**，遇到利空容易产生较大回撤；
3. 和你模块二 44.73 分【黄色审慎区】完全匹配：
> 
> 不是高危红色危机，但不能掉以轻心，适合**降低杠杆、控制单票 / 组合下行敞口，但不需要全面清仓**。

额外观察细节

中证 500 的波动率、波动率斜率的振幅，始终比沪深 300 更大。在你的风控体系里，可以选择：

- 方案 1：以沪深 300 作为全市场基准（当前方案）；
- 方案 2：同时看两个指数，当中证 500 波动率斜率率先突破阈值，作为小盘风险预警。